# DeepFashion Anno Fine DataFrame Builder

This notebook builds a clean dataframe from the `Anno_fine` subset first.

It does four things:

1. reads the split files for `train`, `val`, and `test`
2. aligns attributes, categories, boxes, and landmarks by row order
3. decodes category ids and positive attribute names
4. saves one merged CSV we can reuse later

In [ ]:
from pathlib import Path

import pandas as pd

In [ ]:
PROJECT_ROOT = Path.cwd()
DATA_ROOT = PROJECT_ROOT / "data"
ANNO_FINE = DATA_ROOT / "Anno_fine"
IMAGE_ROOT = DATA_ROOT / "img"
OUTPUT_PATH = DATA_ROOT / "anno_fine_merged.csv"

print(f"Project root: {PROJECT_ROOT}")
print(f"Anno_fine path exists: {ANNO_FINE.exists()}")
print(f"Image root exists: {IMAGE_ROOT.exists()}")

## Read lookup tables

These map numeric ids and attribute columns back to human-readable names.

In [ ]:
def read_category_lookup(path: Path) -> pd.DataFrame:
    rows = []
    with path.open() as f:
        total_categories = int(f.readline().strip())
        f.readline()
        for idx, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            category_name, category_type = line.rsplit(maxsplit=1)
            rows.append(
                {
                    "category_id": idx,
                    "category_name": category_name,
                    "category_type": int(category_type),
                }
            )

    category_lookup = pd.DataFrame(rows)
    assert len(category_lookup) == total_categories
    return category_lookup


def read_attr_lookup(path: Path) -> pd.DataFrame:
    rows = []
    with path.open() as f:
        total_attributes = int(f.readline().strip())
        f.readline()
        for idx, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            attr_name, attr_type = line.rsplit(maxsplit=1)
            rows.append(
                {
                    "attr_index": idx,
                    "attr_name": attr_name,
                    "attr_type": int(attr_type),
                }
            )

    attr_lookup = pd.DataFrame(rows)
    assert len(attr_lookup) == total_attributes
    return attr_lookup


category_lookup = read_category_lookup(ANNO_FINE / "list_category_cloth.txt")
attr_lookup = read_attr_lookup(ANNO_FINE / "list_attr_cloth.txt")

attr_names = attr_lookup["attr_name"].tolist()
attr_columns = [f"attr_{i:02d}" for i in range(1, len(attr_names) + 1)]
attr_name_by_column = dict(zip(attr_columns, attr_names))

display(category_lookup.head())
display(attr_lookup)

## Build one split dataframe

For `Anno_fine`, the `train_attr.txt`, `train_cate.txt`, `train_bbox.txt`, and `train_landmarks.txt` files do not repeat the image names.

They line up by row order with `train.txt`, and the same pattern holds for `val` and `test`.

In [ ]:
def read_lines(path: Path) -> list[str]:
    with path.open() as f:
        return [line.strip() for line in f if line.strip()]


def build_fine_split(split: str) -> pd.DataFrame:
    image_names = read_lines(ANNO_FINE / f"{split}.txt")
    attr_rows = [list(map(int, line.split())) for line in read_lines(ANNO_FINE / f"{split}_attr.txt")]
    cate_rows = [int(line) for line in read_lines(ANNO_FINE / f"{split}_cate.txt")]
    bbox_rows = [list(map(int, line.split())) for line in read_lines(ANNO_FINE / f"{split}_bbox.txt")]
    landmark_rows = [list(map(int, line.split())) for line in read_lines(ANNO_FINE / f"{split}_landmarks.txt")]

    expected_len = len(image_names)
    assert len(attr_rows) == expected_len
    assert len(cate_rows) == expected_len
    assert len(bbox_rows) == expected_len
    assert len(landmark_rows) == expected_len

    attr_df = pd.DataFrame(attr_rows, columns=attr_columns)
    bbox_df = pd.DataFrame(bbox_rows, columns=["x1", "y1", "x2", "y2"])
    landmark_df = pd.DataFrame(
        landmark_rows,
        columns=[f"landmark_{i}_{axis}" for i in range(1, 9) for axis in ["x", "y"]],
    )

    split_df = pd.DataFrame(
        {
            "image_name": image_names,
            "split": split,
            "category_id": cate_rows,
        }
    )

    split_df = pd.concat([split_df, bbox_df, landmark_df, attr_df], axis=1)
    split_df["image_path"] = split_df["image_name"].map(lambda x: str(DATA_ROOT / x))
    split_df["image_exists"] = split_df["image_name"].map(lambda x: (DATA_ROOT / x).exists())

    return split_df


train_df = build_fine_split("train")
val_df = build_fine_split("val")
test_df = build_fine_split("test")

print(train_df.shape, val_df.shape, test_df.shape)
display(train_df.head())

## Merge all splits and decode labels

In [ ]:
anno_fine_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
anno_fine_df = anno_fine_df.merge(category_lookup, on="category_id", how="left")


def positive_attr_names(row: pd.Series) -> list[str]:
    return [attr_name_by_column[col] for col in attr_columns if row[col] == 1]


anno_fine_df["positive_attributes"] = anno_fine_df.apply(positive_attr_names, axis=1)
anno_fine_df["positive_attribute_count"] = anno_fine_df["positive_attributes"].map(len)
anno_fine_df["positive_attributes_str"] = anno_fine_df["positive_attributes"].map(lambda x: "|".join(x))

print(f"Merged rows: {len(anno_fine_df):,}")
print(anno_fine_df["split"].value_counts())
print(f"Missing category names: {anno_fine_df['category_name'].isna().sum()}")
print(f"Missing local image files: {(~anno_fine_df['image_exists']).sum()}")

display(anno_fine_df[["image_name", "split", "category_id", "category_name", "positive_attributes_str"]].head())

## Quick checks

In [ ]:
summary = {
    "rows": len(anno_fine_df),
    "train_rows": int((anno_fine_df["split"] == "train").sum()),
    "val_rows": int((anno_fine_df["split"] == "val").sum()),
    "test_rows": int((anno_fine_df["split"] == "test").sum()),
    "avg_positive_attributes": round(float(anno_fine_df["positive_attribute_count"].mean()), 2),
    "unique_categories": int(anno_fine_df["category_name"].nunique()),
}
summary

In [ ]:
anno_fine_df[["category_name", "split"]].value_counts().head(15)

In [ ]:
attr_frequency = anno_fine_df[attr_columns].sum().sort_values(ascending=False)
top_attrs = pd.DataFrame(
    {
        "attr_column": attr_frequency.index,
        "count": attr_frequency.values,
        "attr_name": [attr_name_by_column[col] for col in attr_frequency.index],
    }
)
top_attrs.head(15)

## Save the merged dataframe

This gives us one reusable flat file before we move on to the larger `Anno_coarse` dataset.

In [ ]:
anno_fine_df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved merged dataframe to: {OUTPUT_PATH}")